# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
before = len(df)
df = df.drop_duplicates()
log(1, 'dropped duplicate rows', before - len(df))

[1] dropped duplicate rows (15 row(s))


### TODO 2 — clean `price` -> float

In [4]:
df['price'] = df['price'].astype(str).str.replace('$', '', regex=False).astype(float)
log(2, 'stripped $ and cast price to float', len(df))

[2] stripped $ and cast price to float (300 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
# TODO
before = len(df)
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')
df = df.dropna(subset=['qty'])
df = df[df['qty'] > 0]
log(3, 'cleaned qty', before - len(df))

[3] cleaned qty (25 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
# TODO: inspect the variants, build a mapping dict, apply it, log the collapse
print(df['item'].value_counts())
ITEM_MAP = {
    'cheese burger': 'Cheeseburger',
    'foam finger': 'Foam Finger',
    'rain poncho': 'Rain Poncho',
}
changed = df['item'].isin(ITEM_MAP.keys()).sum()
df['item'] = df['item'].replace(ITEM_MAP)
log(4, '6 item spellings -> 3', changed)


item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[4] 6 item spellings -> 3 (126 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [7]:
print(df['category'].value_counts())
CAT_MAP = {'Apparel': 'Merch', 'food': 'Food', 'rain-gear': 'RainGear'}
changed = df['category'].isin(CAT_MAP.keys()).sum()
df['category'] = df['category'].replace(CAT_MAP)
log(5, 'merged Apparel into Merch (business decision); unified food/rain-gear spelling', changed)

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[5] merged Apparel into Merch (business decision); unified food/rain-gear spelling (132 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [8]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
# TODO: assert something about item
assert df['item'].nunique() == 3
# TODO: assert something about category
assert df['category'].nunique() == 3
print('clean:', df.shape)

log(6, 'proven clean', len(df))


clean: (275, 5)
[6] proven clean (275 row(s))


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [9]:
# TODO
df['revenue'] = df['qty'] * df['price']
print(df.groupby('category')['revenue'].sum().sort_values(ascending=False).round(2))
print('Total:', df['revenue'].sum().round(2))
log(7, 'calculated revenue', len(df))


category
Food        1656.0
Merch       1572.0
RainGear    1512.0
Name: revenue, dtype: float64
Total: 4740.0
[7] calculated revenue (275 row(s))


**What I would tell the vendor:** I'd tell the vendor to stock slightly more Food, since it's the top category at \$1,656, but all three are within about \$150 of each other, so the difference is small.

### TODO 8 — read back your log

In [10]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,1,dropped duplicate rows,15
1,2,stripped $ and cast price to float,300
2,3,cleaned qty,25
3,4,6 item spellings -> 3,126
4,5,merged Apparel into Merch (business decision);...,132
5,6,proven clean,275
6,7,calculated revenue,275


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

a. The step that changes revenue most is the first step (dropping duplicates). Before dropping duplicates the revenue is \$4,852.50. After dropping duplicates, the revenue is \$4,594.50, which is \$258.00 less than the initial revenue before the cleaning.

b. Merging RainGear into Merch would make Merch revenue \$3,084, while Food is \$1,656. Keeping them separate makes Food the lead in revenue. I would keep rain gear separate because it is a different product than merch and apparel.